## Generation of Protein Vector Given a 2D Sequence

Import Packages:

In [2]:
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F

import re

from typing import Union

import transformers
from transformers import AutoTokenizer, AutoModel


import requests
import esm

url = "https://raw.githubusercontent.com/schwallergroup/ai4chem_course/main/notebooks/06%20-%20Generative%20Models%202/utils.py"

r = requests.get(url)
with open("utils.py", "wb") as f:
    f.write(r.content)

url = "https://raw.githubusercontent.com/sch20-%20Generative%20Models%202/pretrained.zinc.rnn.pth"

r = requests.get(url)
with open("pretrained.zinc.rnn.pth", "wb") as f:
    f.write(r.content)

import torch.nn as nn

from tscales_bert_cls import TScalesBERTEncoder, encode_tscales_cls




/Users/michal/Library/CloudStorage/OneDrive-epfl.ch/AI-for-Chemistry-Project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the dataset:

In [3]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")

Generate t-scale for protein sequences:

In [4]:
encoder = TScalesBERTEncoder(d_model=256, nhead=8, num_layers=4)
tscales_cls = encode_tscales_cls(df["Protein sequence"], encoder)
df["tscales_cls"] = list(tscales_cls)

/Users/michal/Library/CloudStorage/OneDrive-epfl.ch/AI-for-Chemistry-Project/tscales_bert_cls.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer,


Using Evolutionary Scale Modeling (ESM-2) Embeddings

In [5]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

In [6]:
def get_esm_embedding(seq):
    data = [("protein", seq)]

    _, _, tokens = batch_converter(data)

    with torch.no_grad():
        results = model(tokens, repr_layers=[33])

    # layer 33 = final representation
    reps = results["representations"][33]

    # remove special tokens and mean pool
    embedding = reps[0, 1:-1].mean(0)

    return embedding.numpy()


df["esm"] = df["Protein sequence"].apply(get_esm_embedding)

Encode SMILES of chromophores using pre-trained chemBERTa RNN

In [7]:
#assignment of canonical SMILES of ligand
smiles_dict = {
    "NRQ": r"CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRQ": r"NC(=O)CCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "NRP": r"CC(C)CC(=N)C1=NC(=C/c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "CH6": r"CSCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRO": r"[C@@H](O)[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "5SQ": r"N[C@@H](Cc1c[nH]cn1)C2=NC(=C\c3ccc(O)cc3)/C(=O)N2CC(O)=O",
    "4M9": r"NC(=O)CCC(=N)C1=NC(=C\c2c[nH]c3ccccc23)/C(=O)N1CC(O)=O",
    "CR2": r"NCC1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "OFM": r"C[C@H]1O[C@@](O)(N=C1C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O)[C@@H](N)Cc4ccccc4",
    "CR8": r"N[C@@H](Cc1[nH]cnc1)c2nc(C=C3C=CC(=O)C=C3)c([O-])n2CC(O)=O",
    "CFY": r"N[C@@H](Cc1ccccc1)[C@@]2(O)SCC(=N2)C3=NC(=C\c4ccc(O)cc4)/C(=O)N3CC(O)=O",
    "OIM": r"CC[C@H](C)[C@H](N)[C@@]1(O)O[C@H](C)C(=N1)C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O",
    "CH7": r"OC(=O)CN1C(=O)C(=C/c2ccc(O)cc2)/N=C1C3=NCCCC3",
    "GYS": r"N[C@@H](CO)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "WCR": r"C[C@@]1(O)NC(=C\c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "DYG": r"N[C@@H](CC(O)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FAD": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C",
    "PIA": r"C[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "BLR": r"CC1=C(C=C)C(/NC1=O)=C/c2[nH]c(Cc3[nH]c(\C=C4/NC(=O)C(=C4C)C=C)c(C)c3CCC(O)=O)c(CCC(O)=O)c2C",
    "CRF": r"C[C@@H](O)[C@H](N)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "NYG": r"N[C@@H](CC(N)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FMN": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C",
    "B2H": r"C[C@@H](O)[C@H](N)c1nc(Cc2c[nH]c3ccccc23)c(O)n1CC(O)=O",
    "SWG": r"N[C@@H](CO)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "CSH": r"N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC(O)=O",
    "BJF": r"CC(C)C[C@H](N)c1nc(CC(C)C)c(O)n1CC(O)=O",
    "GYC": r"N[C@@H](CS)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CCY": r"N[C@@H](CS)[C@H]1N[C@@H](Cc2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CR7": r"NCCCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O"
}   
df["smiles"] = df["Chromophore/ligand"].str.strip().str.upper().map(smiles_dict)
from transformers import AutoTokenizer, AutoModel


model_name = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def smiles_to_vector(smiles):
    if not isinstance(smiles, str):
        return None

    inputs = tokenizer(smiles, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # CLS token embedding = molecular representation
    pooled = outputs.last_hidden_state.max(dim=1).values.squeeze()

    return pooled.numpy()


df["smiles_vectors"] = df["smiles"].apply(smiles_to_vector)



Concatenation of sequence vectors obtained with ESM, ligand SMILES vectors obtained with chemBERTa, and predictor values (Stokes shift, EC, quantum yield, protein mass) 

Define 2 model

In [8]:
# ── Model 1 (existing — no changes) ──────────────────────────────────────────
def concat_model1(row):
    return np.concatenate([
        row["esm"],                    # 1280-d
        row["smiles_vectors"],         # 768-d
        np.array([
            row["Stokes shift"],      # Optional
            row["kDa"]
        ])
    ])



def concat_model2(row):
    return np.concatenate([
        row["tscales_cls"],            # 256-d
        row["smiles_vectors"],         # 768-d
        np.array([
            row["Stokes shift"],       # Optional
            row["kDa"]
        ])
    ])



Normalize and split

In [9]:
# Normalize FIRST
scaler = StandardScaler()
num_cols = ["Stokes shift", "kDa"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols]  = scaler.transform(test_df[num_cols])



Building input vectors

In [10]:
# THEN build input vectors on train_df and test_df separately
train_df["inputs_model1"] = train_df.apply(concat_model1, axis=1)
test_df["inputs_model1"]  = test_df.apply(concat_model1, axis=1)

train_df["inputs_model2"] = train_df.apply(concat_model2, axis=1)
test_df["inputs_model2"]  = test_df.apply(concat_model2, axis=1)